In [12]:
import sys, os
sys.path.insert(0, '/home/nvidia/24BCE1793/fignn_cni__project')
os.chdir('/home/nvidia/24BCE1793/fignn_cni__project')

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

GPU: NVIDIA A100-SXM4-80GB
VRAM: 85 GB


In [13]:
from src.preprocessing.preprocess import process_file
from src.preprocessing.plv import batch_compute_plv
from src.preprocessing.graph_builder import build_graph_dataset
from src.training.train import run_training
from src.evaluation.evaluate import run_full_evaluation, plot_loss_curve

In [14]:
DATA_ROOT = "/lp-dev/24BCE1793/data/raw"
TASKS     = {"zeroBACK": 0, "oneBACK": 1, "twoBACK": 2}
SESSIONS  = ["ses-S1", "ses-S2", "ses-S3"]
SUBJECTS  = [f"sub-{i:02d}" for i in range(1, 30)]  # all 29

In [15]:
set_files = []
for sub in SUBJECTS:
    for ses in SESSIONS:
        for task, label in TASKS.items():
            path = os.path.join(DATA_ROOT, sub, ses, "eeg", f"{task}.set")
            if os.path.exists(path):
                set_files.append((path, label))
            else:
                print(f"[skip] {sub}/{ses}/{task}")

print(f"Found {len(set_files)} files")
print(f"Expected: {len(SUBJECTS)*3*3} = 261 files")

Found 261 files
Expected: 261 = 261 files


In [ ]:
all_graphs = []

for i, (set_path, label) in enumerate(set_files):
    sub = set_path.split('/')[-4]
    task = set_path.split('/')[-1]
    print(f"[{i+1}/{len(set_files)}] {sub} {task}")
    
    try:
        processed = process_file(set_path, label)
        plv_list  = batch_compute_plv(processed["band_epochs"], verbose=False)
        graphs    = build_graph_dataset(processed, plv_list)
        all_graphs.extend(graphs)
        print(f"  +{len(graphs)} graphs | total: {len(all_graphs)}")
    except Exception as e:
        print(f"  ERROR: {e} — skipping")
        continue

print(f"\nDataset complete: {len(all_graphs)} graphs")
for lv, ln in [(0,"zero"),(1,"one"),(2,"two")]:
    c = sum(1 for g in all_graphs if g.y.item()==lv)
    print(f"  label {lv} ({ln}BACK): {c} graphs")

In [9]:
import pickle
os.makedirs("/lp-dev/24BCE1793/data/processed", exist_ok=True)
with open("/lp-dev/24BCE1793/data/processed/all_graphs_29sub.pkl", "wb") as f:
    pickle.dump(all_graphs, f)
print("Dataset saved")

NameError: name 'all_graphs' is not defined

In [ ]:
model, train_hist, val_hist = run_training(
    all_graphs,
    n_epochs   = 100,
    batch_size = 64,
    lr         = 1e-3,
)

In [ ]:
plot_loss_curve(train_hist, val_hist)
from IPython.display import Image
Image('/lp-dev/24BCE1793/outputs/plots/loss_curve.png')

In [ ]:
device = torch.device("cuda")
results = run_full_evaluation(model, all_graphs, device)
print(f"Silhouette: {results['silhouette']:.4f}")
print(f"Band weights: {results['band_weights']}")

In [ ]:
from IPython.display import Image, display
display(Image('/lp-dev/24BCE1793/outputs/plots/tsne_latent_space.png'))
display(Image('/lp-dev/24BCE1793/outputs/plots/band_importance.png'))